In [1]:
import uuid
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore

# UpstageDocumentParseLoader MD 파일 불러오기

In [2]:
import ast
from langchain.schema import Document
from langchain.vectorstores import Chroma
from langchain_upstage import UpstageEmbeddings

file_path = "C:/Projects/CRA/UpstageDocumentParserMD.txt"

In [3]:
# 텍스트 파일 파싱 함수
def parse_text_file(file_path):
    documents = []
    with open(file_path, "r", encoding='utf-8') as file:
        raw_data = file.read()
    
    # page_content로 분리
    entries = raw_data.split("page_content=")
    
    for entry in entries[1:]:  # 첫 번째 빈 항목 제외
        # content와 metadata 분리
        content, metadata = entry.split("metadata=", 1)
        # 따옴표와 공백 제거
        content = content.strip().strip("'")
        # 메타데이터 문자열을 딕셔너리로 변환
        metadata = ast.literal_eval(metadata.strip())
        
        # Document 객체 생성
        doc = Document(page_content=content, metadata=metadata)
        documents.append(doc)
    
    return documents

In [4]:
def remove_coordinates(documents):
    updated_documents = []
    for doc in documents:
        # Document 객체의 metadata 속성에 접근
        metadata = doc.metadata.copy()
        if 'coordinates' in metadata:
            del metadata['coordinates']
        
        # 새로운 Document 객체 생성
        updated_doc = Document(
            page_content=doc.page_content,
            metadata=metadata
        )
        updated_documents.append(updated_doc)
    return updated_documents


In [5]:
documents = parse_text_file(file_path)

In [6]:
documents = remove_coordinates(documents)

In [38]:
documents[37]

Document(metadata={'page': 38}, page_content='# 8. 입학년도별 교과과정 안내 # ❚ 2024학년도 입학자 교과과정 1. 졸업 기준 | 구 분 | 구 분 | 이 수 과 목 | 이 수 과 목 | 주 요 사 항 |\n| --- | --- | --- | --- | --- |\n| 교 양 | 공통필수 | 8개 교과목 | 8개 교과목 | 세종인을위한진로설계, 세종인을위한전공탐색, 창업과기업가정신1, 문제해결을위한글쓰기와발표, 서양철학:쟁점과토론, 우주자연인간, 취창업과진로설계, 대학영어 |\n| 교 양 | 계 균형교양 열 별 학문기초 필 교양 수 | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) |\n| 교 양 | 계 균형교양 열 별 학문기초 필 교양 수 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 |\n| 전공 | 전공 | 구 분 | 내 용 | 내 용 |\n| 전공 | 전공 | 단일전공 이수시 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 |\n| 전공 | 전공 | 복수전공 이수시 (연계·융합 전공포함) | -전필 : 15 학점 -전선 : 24 학점 -합계 : 39 학점(주전공, 복수전공 각각이수) ※ 건축학전공 이수자는(6.전공·복수전공·부전공·제2전공 신청 및 이수안내 참조) ※ 교직과정 이수자가 교직복수전공시 주, 복수전공 각각 50학점 이상 이수 ※ 법학부, 호텔외식관광프랜차이즈경영학과와 국방시스템공학과, 항공 시스템공학과 등 계약학과의 복수전공에 관한 사항은 별도 규정에 따름 | -전필 : 15 학점 -전선 : 24 학점 -합계 : 39 학점(주전공, 복수전공 각각이수) ※ 

In [72]:
import json
from langchain.schema import Document

# Document 객체를 JSON으로 변환하여 저장
def save_documents_json(documents, file_path):
    docs_dict = [{
        "page_content": doc.page_content,
        "metadata": doc.metadata
    } for doc in documents]
    
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(docs_dict, f, ensure_ascii=False, indent=2)

# JSON에서 Document 객체로 로드
def load_documents_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs_dict = json.load(f)
    
    return [Document(
        page_content=doc["page_content"],
        metadata=doc["metadata"]
    ) for doc in docs_dict]

# 사용 예시
file_path = "courseCatalog.json"
save_documents_json(documents, file_path)
#loaded_docs = load_documents_json(file_path)


# MultiVector를 위한 Chroma DB와 LocalDB 생성

In [8]:
VECTOR_DB_PATH = "./MultiVector/multiVectorDB"
DOCS_DB_PATH = "./MultiVector/docsDB"

vectorstore = Chroma(
    persist_directory=VECTOR_DB_PATH,
    collection_name="multiVectorStore",
    embedding_function=UpstageEmbeddings(model="embedding-passage"),
)
# 문서 저장소 초기화 (로컬 파일 시스템 사용)
docstore = LocalFileStore(DOCS_DB_PATH)

id_key = "doc_id"

# 검색기 (시작 시 비어 있음)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=docstore,
    id_key=id_key,
)

C:\Users\catus\AppData\Local\Temp\ipykernel_18172\64686854.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [9]:
# 문서 ID를 생성합니다.
doc_ids = [str(uuid.uuid4()) for _ in documents]

# 두개의 생성된 id를 확인합니다.
doc_ids

['68911863-b27a-47a5-b435-4087d9ec4661',
 'bae90340-ee9f-4854-bbb3-85b184c4d4bb',
 'a798015e-e2c6-44ac-9bb4-f198ab0eae4e',
 'eb6e131d-bce9-4713-9922-5b08041d134c',
 'afabada6-fc85-4ebe-b111-88a7f2138ffa',
 '35146b7c-73fe-4e3b-8568-a0be62dd2d89',
 '4aa0d28d-53b4-4871-9d3d-7966062f014b',
 'a54deb27-a3c6-486d-8c18-26ae2a0e6ff8',
 'e7f03ec5-9f92-44f6-81cc-d9d1c05d0030',
 '58aef24b-82a8-44a7-9aba-161e0724b88e',
 '5afc9ca3-43f1-4edd-b606-6247d7dfd3fb',
 'fb5e6718-7d94-4be0-9e65-f1f10934f688',
 '1257a87f-068a-4ee2-9004-79e4e5cdd568',
 '7f402b0a-d808-4906-a0f5-449b2230afdf',
 'edb4e3f9-7874-4ea8-be4a-e5921ddda524',
 '19565a4d-7aba-4126-8f84-718d24dafb10',
 '2b9b3ae5-9a6f-472a-955c-c43955083100',
 'e358f6cc-0954-452d-b409-ac07f10645d5',
 'af371f6d-2ad2-4c58-b393-fa7fb644cd91',
 'a6ed16fe-0be5-4a94-83fe-3eed640d6949',
 '7c736a95-9586-403e-97b8-2f048fc46021',
 'b02db2a2-1033-46b2-9416-b5b9f637e95c',
 'a6254de5-c74d-41dd-ac5f-1103a8877bda',
 '71180de8-f041-40af-9472-3742a4aa7f7b',
 'b30087ff-a5c2-

## Retriever Settings(ParentDocumentRetriever)

In [10]:
# RecursiveCharacterTextSplitter 객체를 생성합니다.
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)

# 더 작은 청크를 생성하는 데 사용할 분할기
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

In [11]:
parent_docs = []

for i, doc in enumerate(documents):
    # 현재 문서의 ID를 가져옵니다.
    _id = doc_ids[i]
    # 현재 문서를 하위 문서로 분할
    parent_doc = parent_text_splitter.split_documents([doc])

    for _doc in parent_doc:
        # metadata에 문서 ID 를 저장
        _doc.metadata[id_key] = _id
    parent_docs.extend(parent_doc)

In [12]:
# 생성된 Parent 문서의 메타데이터를 확인합니다.
parent_docs[0].metadata

{'page': 1, 'doc_id': '68911863-b27a-47a5-b435-4087d9ec4661'}

In [13]:
child_docs = []
for i, doc in enumerate(documents):
    # 현재 문서의 ID를 가져옵니다.
    _id = doc_ids[i]
    # 현재 문서를 하위 문서로 분할
    child_doc = child_text_splitter.split_documents([doc])
    for _doc in child_doc:
        # metadata에 문서 ID 를 저장
        _doc.metadata[id_key] = _id
    child_docs.extend(child_doc)

In [14]:
# 생성된 Child 문서의 메타데이터를 확인합니다.
child_docs[0].metadata

{'page': 1, 'doc_id': '68911863-b27a-47a5-b435-4087d9ec4661'}

In [15]:
print(f"분할된 parent_docs의 개수: {len(parent_docs)}")
print(f"분할된 child_docs의 개수: {len(child_docs)}")

분할된 parent_docs의 개수: 203
분할된 child_docs의 개수: 916


In [16]:
# 벡터 저장소에 parent + child 문서를 추가
retriever.vectorstore.add_documents(parent_docs)
print("end parent\n")
retriever.vectorstore.add_documents(child_docs)
print("end child\n")

# docstore 에 원본 문서를 저장
retriever.docstore.mset(list(zip(doc_ids, documents)))

end parent

end child



# 여러가지 검색기능

In [20]:
# vectorstore의 유사도 검색을 수행합니다.
relevant_chunks = retriever.vectorstore.similarity_search(
    "24학번은 뭐 들어야돼?"
)
print(f"검색된 문서의 개수: {len(relevant_chunks)}")

for chunk in relevant_chunks:
    print(chunk.page_content, end="\n\n")
    print(">" * 100, end="\n\n")

검색된 문서의 개수: 4
< 차 례 >
2024학년도 학사일정 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 3
수강신청 및 학사제도 주요 변경안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 4
1. 수강신청 안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 5
수강신청 일정, 관심과목담기, 수강신청 절차 및 유의사항, 수강신청 학점이월제도, 폐강시점 및
기준 안내, 수강변경, 수강과목 철회, 수강과목 유의사항[일괄수강신청 과목, 수강제한 과목,
대학원 석사과정 교과목 수강안내, 학석사 연계과목 수강안내, 타 학과 전공선택 인정교과목 수강안내]
2. 수강 관련 일반 사항 안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 18
강의시간, 강의실, 수강대상 및 유의사항, 교수-자녀 간 강의수강 제한, 학사경고자대상 프로그램
운영 및 미이수자 학점제한제도, 졸업인증제(영어, 고전독서, 소프트웨어코딩, 한국어, 공학교육)
3. 교과목 수강안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 22
세종인을위한진로설계∙전공탐색, 대학영어, English Listening/Reading Practice,
서양철학:쟁점과토론/문제해결을위한글쓰기와발표, 창업과기업가정신, 취창업과진로설계,
코딩교과목, 인공지능과빅데이터, 세종사회봉사1·2, 블렌디드 강의, 본교 e-러닝 강의

In [31]:
from langchain.retrievers.multi_vector import SearchType

# 검색 유형을 MMR(Maximal Marginal Relevance)로 설정
retriever.search_type = SearchType.mmr

# 관련 문서 전체를 검색
print(retriever.invoke("24학번은 뭐 들어야돼??")[3].page_content)

# < 2024학년도 1학기 강의시간표 차례 > 공 통 인공지능융합대학
‣공 통 교 양 필 수 ․ ․ ․ ․ ․ ․ ․ ․ 119 ‣인 공 지 능 데 이 터 사 이 언 스 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 152
‣교 양 필 수 ․ ․ ․ ․ ․ ․ ․ ․ 123 데 이 터 사 이 언 스 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 152
‣균 형 교 양 필 수 ․ ․ ․ ․ ․ ․ ․ ․ 123 인 공 지 능 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 153
‣학 문 기 초 교 양 필 수 ․ ․ ․ ․ ․ ․ ․ ․ 123 ‣A I 로 봇 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 153
‣교 양 선 택 ․ ․ ․ ․ ․ ․ ․ ․ 125 지 능 기 전 공 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 153
‣무 관 후 보 생 교 육 ․ ․ ․ ․ ․ ․ ․ ․ 129 무 인 이 동 체 공 학 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 154
‣교 직 ․ ․ ․ ․ ․ ․ ․ ․ 129 스 마 트 기 기 공 학 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 154
‣창 의 소 프 트 학 부 ․ ․ ․ ․ ․ ․ ․ ․ 155
인문사회/경상/자연생명/IT/공과계열 ․ ․ ․ ․ ․ ․ ․ ․ 130 디 자 인 이 노 베 이 션 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 155
만 화 애 니 메 이 션 텍 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 156
인문과학대학
‣국 어 국 문 학 과 ․ ․ ․ ․ ․ ․ ․ 131 공과대학
‣국 제 학 부 ․ ․ ․ ․ ․ ․ ․ 132 ‣건 축 공 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 156
영 어 영 문 학 전 공 ․ ․ ․ ․ ․ ․ ․ 132 건 축 공 학 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 157
일 어 일 문 학 전 공 ․ ․ ․ ․ ․ ․ ․ 132 ‣건 축 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 157
중 국 통 상 학 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 133 건 축 학 전 공 ․ ․ ․ ․ ․ ․ ․ ․ 157
‣역 사 학 과 ․ ․ ․ ․ ․ ․ ․ ․ 133 ‣건 설

In [33]:
from langchain.retrievers.multi_vector import SearchType

# 검색 유형을 similarity_score_threshold로 설정
retriever.search_type = SearchType.similarity_score_threshold
retriever.search_kwargs = {"score_threshold": 0.1}

# 관련 문서 전체를 검색
print(retriever.invoke("24학번은 뭐 들어?")[0].page_content)

< 차 례 >
2024학년도 학사일정 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 3
수강신청 및 학사제도 주요 변경안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 4
1. 수강신청 안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 5
수강신청 일정, 관심과목담기, 수강신청 절차 및 유의사항, 수강신청 학점이월제도, 폐강시점 및
기준 안내, 수강변경, 수강과목 철회, 수강과목 유의사항[일괄수강신청 과목, 수강제한 과목,
대학원 석사과정 교과목 수강안내, 학석사 연계과목 수강안내, 타 학과 전공선택 인정교과목 수강안내]
2. 수강 관련 일반 사항 안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 18
강의시간, 강의실, 수강대상 및 유의사항, 교수-자녀 간 강의수강 제한, 학사경고자대상 프로그램
운영 및 미이수자 학점제한제도, 졸업인증제(영어, 고전독서, 소프트웨어코딩, 한국어, 공학교육)
3. 교과목 수강안내 ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ ․ 22
세종인을위한진로설계∙전공탐색, 대학영어, English Listening/Reading Practice,
서양철학:쟁점과토론/문제해결을위한글쓰기와발표, 창업과기업가정신, 취창업과진로설계,
코딩교과목, 인공지능과빅데이터, 세종사회봉사1·2, 블렌디드 강의, 본교 e-러닝 강의, PBL 과목,
FL 과

# 요약 정보 저장

In [39]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

In [42]:
summary_chain = (
    {"doc": lambda x: x.page_content}
    # 문서 요약을 위한 프롬프트 템플릿 생성
    | ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert in summarizing documents in Korean."),
            (
                "user",
                "Summarize the following documents in 3 sentences in bullet points format.\n\n{doc}",
            ),
        ]
    )
    # OpenAI의 ChatGPT 모델을 사용하여 요약 생성
    | ChatOpenAI(temperature=0, model="gpt-4o-mini")
    | StrOutputParser()
)

In [43]:
# 문서 배치 처리
summaries = summary_chain.batch(documents, {"max_concurrency": 10})

In [45]:
len(summaries)

119

In [48]:
# 원본 문서의 내용을 출력합니다.
print(documents[41].page_content, end="\n\n")
# 요약을 출력합니다.
print("[요약]")
print(summaries[41])

# ❚ 2023학년도 입학자 교과과정 # 1. 졸업 기준 | 구 분 | 구 분 | 구 분 | 이 수 과 목 | 이 수 과 목 | 주 요 사 항 |
| --- | --- | --- | --- | --- | --- |
| 교 양 | 공통필수 | 공통필수 | 8개 교과목 | 8개 교과목 | 신입생세미나A, 신입생세미나B, 창업과기업가정신1, 문제해결을위한글쓰기와발표, 서양철학:쟁점과토론, 우주자연인간, 취창업과진로설계, 대학영어 |
| 교 양 | 계 열 별 필 수 | 균형교양 | 자신의 소속계열과 다른 2개 영역에서 6학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 2개 영역에서 6학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 2개 영역에서 6학점 선택 이수 (학생자율선택) |
| 교 양 | 계 열 별 필 수 | 학문기초 교양 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 | 단과대학 또는 학과에 따라 지정된 과목 선택 이수 |
| 전공 | 전공 | 전공 | 구 분 | 내 용 | 내 용 |
| 전공 | 전공 | 전공 | 단일전공 이수시 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 |
| 전공 | 전공 | 전공 | 복수전공 이수시 (연계·융합 전공포함) | -전필 : 15 학점 -전선 : 24 학점 -합계 : 39 학점(주전공, 복수전공 각각이수) ※ 건축학전공 이수자는(6.전공·복수전공·부전공·제2전공 신청 및 이수안내 참조) ※ 교직과정 이수자가 교직복수전공시 주, 복수전공 각각 50학점 이상 이수 ※ 법학부, 호텔외식관광프랜차이즈경영학과와 국방시스템공학과, 항공 시스템공학과 등 계약학과의 복수전공에 관한 사항은 별도 규정에 따름 | -전필 : 15 학점 -전선 : 24 학점 -합계 : 39 학점(주전공, 복수전공 각각이수) ※ 건축학전공 이수자는(6.전공·복수전공·부전공·제2전공 신청 및 이수안내 참조) ※ 교직과정 

In [50]:
VECTOR_DB_PATH = "./MultiVectorSummary/multiVectorDB"
DOCS_DB_PATH = "./MultiVectorSummary/docsDB"

# 요약 정보를 저장할 벡터 저장소를 생성합니다.
summary_vectorstore = Chroma(
    persist_directory=VECTOR_DB_PATH,
    collection_name="summaries",
    embedding_function=UpstageEmbeddings(model="embedding-passage"),
)

# 부모 문서를 저장할 저장소를 생성합니다.
docstore = LocalFileStore(DOCS_DB_PATH)

# 문서 ID를 저장할 키 이름을 지정합니다.
id_key = "doc_id"

# 검색기를 초기화합니다. (시작 시 비어 있음)
retriever = MultiVectorRetriever(
    vectorstore=summary_vectorstore,  # 벡터 저장소
    byte_store=docstore,  # 바이트 저장소
    id_key=id_key,  # 문서 ID 키
)
# 문서 ID를 생성합니다.
doc_ids = [str(uuid.uuid4()) for _ in documents]



In [51]:
summary_docs = [
    # 요약된 내용을 페이지 콘텐츠로 하고, 문서 ID를 메타데이터로 포함하는 Document 객체를 생성합니다.
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

In [52]:
# 요약본의 문서의 개수
len(summary_docs)

119

In [68]:
summary_docs[37]

Document(metadata={'doc_id': 'efa9b8c4-96a2-4728-a0dd-96b5aea5eeb0'}, page_content='- 2024학년도 입학자 교과과정은 졸업을 위해 130학점(건축학과는 168학점) 이수해야 하며, 공통필수 과목으로 8개 교과목을 반드시 이수해야 한다.\n- 전공 이수 시 단일전공은 학과에 따라 다르며, 복수전공은 전필 15학점, 전선 24학점 총 39학점을 이수해야 한다.\n- 졸업 인증제는 영어, 고전 독서, 소프트웨어 코딩 중 2개 이상 통과해야 하며, 예체능대학은 졸업작품 이수로 대체 가능하다.')

In [53]:
retriever.vectorstore.add_documents(
    summary_docs
)  # 요약된 문서를 벡터 저장소에 추가합니다.

# 문서 ID와 문서를 매핑하여 문서 저장소에 저장합니다.
retriever.docstore.mset(list(zip(doc_ids, documents)))

In [70]:
# 유사도 검색을 수행합니다.
result_docs = summary_vectorstore.similarity_search(
    "24학번은 졸업하려면 뭐해야돼?"
)
# 1개의 결과 문서를 출력합니다.
print(result_docs[3].page_content)

- 학사경고를 받은 학생은 방학 중 학습능력향상 프로그램을 이수해야 하며, 미이수 시 수강신청 가능 학점이 15학점으로 제한된다.
- 졸업인증제는 2023학년도 이후 입학자에게 영어, 고전독서, 소프트웨어코딩 중 2개 이상 통과해야 하며, 예체능대학은 졸업작품 이수로 대체된다.
- 영어졸업인증은 외부 공인영어시험에서 기준 점수를 취득해야 하며, 영어영문학 전공자는 더 높은 기준을 적용받는다.


In [71]:
# 관련된 문서를 검색하여 가져옵니다.
retrieved_docs = retriever.invoke("24학번은 뭐 들어야돼?")
print(retrieved_docs[3])

page_content='8. 인공지능과빅데이터(기필, 3학점, 본교e-러닝) 가. 수강대상 : 2022학년도 이후 입학자 2학년 대상학과 | 분반 | 수강 대상 학과 |
| --- | --- |
| 1 | 국어국문, 국제학부, 역사학, 교육학2 |
| 2 | 행정학, 법학, 디자인이노베이션학, 만화애니메이션텍2 |
| 3 | 물리천문학, 식품생명공학, 바이오산업자원공학, 스마트생명산업융합학2 |
| 4 | 전자정보통신공학, 반도체시스템공학2 |
| 5 | 컴퓨터공학, 소프트웨어학2 |
| 6 | 지능기전, 인공지능학2 |
| 7 | 정보보호, 데이터사이언스학, 화학, 건설환경공학, 항공시스템학과2 대상학과 |
| 8 | 재수강반 (해당 과목을 필수로 이수해야하는 학생만 수강가능) |
 나. 반드시 수강대상 학과 분반에 수강신청 하시기 바랍니다.
다. 관련문의 : 교양코딩실 (대양AI센터 409호, 02-6935-2535) 9. 세종사회봉사1·2 과목 (교선, 1학점) 가. 개설목적 : 사회봉사에 대한 이론과 실습을 통해 사회문제에 대한 현실 인식을 높이며, 공동체 의식을
배양하는 것을 목적으로 한다. 나. 수강대상 1) 2020~2023학년도 입학자의 경우 세종사회봉사1과 세종사회봉사2는 교양선택 과목으로 인정된다.
2) 2012~2019학년도 입학자 - 가) 세종사회봉사1을 재수강하는 경우 교양필수, 세종사회봉사2는 교양선택 과목으로 인정된다.
- 나) 세종사회봉사1은 졸업 필수 조건 교과목으로, 졸업 전 반드시 이수하여야 한다.
- 다) 세종사회봉사1은 세종사회봉사2의 선이수 과목으로, 세종사회봉사1과 2를 동시 수강할 수 없다.
 3) 2011학년도 이전 입학자의 경우 세종사회봉사1과 세종사회봉사2는 교양선택 과목으로 인정된다.
다. 세종사회봉사 교과목 운영 안내 1) 세종사회봉사1·2 : 집현캠퍼스 온라인강의 + 30시간 이상 봉사활동 + 집현캠퍼스 ‘과제’ 제출
※ 세종사회봉사1 과목은 학점 제한 없이 수강신청 가능하며 제출 방법 등 자세한 사항은 